### Tools
Models can request to call tools that perform tasks such as 

1. fetching data from a database, 
2. searching the web, 
3. or running code. 

Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")

print(response.content)

In [18]:
from langchain_groq import ChatGroq
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

@tool
def get_latest_news(location:str)->str:
    """Get the latest news at a location"""
    return f"Here are the latest news in {location}"

model = ChatGroq(model="openai/gpt-oss-20b")
# bind_tools allows the model to call the tools when needed
# We can also use create_agent to create an agent that can call the tools when needed
model_with_tools=model.bind_tools([get_weather, get_latest_news])

response = model_with_tools.invoke("What's the latest news in Boston?")
print(response.tool_calls)
print('*' * 10)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

[{'name': 'get_latest_news', 'args': {'location': 'Boston'}, 'id': 'fc_0e87ca2c-f43f-4581-a6a0-7944312db3b7', 'type': 'tool_call'}]
**********
Tool: get_latest_news
Args: {'location': 'Boston'}


### Tool Execution Loops

In [47]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It’s sunny in Boston.


In [44]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_ea00116f-909c-4cdd-8e94-8d0801bea1d2', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 150, 'total_tokens': 182, 'completion_time': 0.034606992, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007760062, 'prompt_tokens_details': None, 'queue_time': 0.210028919, 'total_time': 0.042367054}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20e1e-3cd3-72e6-a977-049102456bb9-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_ea00116f-909c-4cdd-8e94-8d0801bea1d2', 'type': 'tool_cal